# MASLD-Bench · 01 — Extra Corpus Builders (Drive-synced edition)



Builds the corpora that need Colab RAM (Tahoe perturbation). Your local folders

sync via Drive for Desktop and mount at `/content/drive/Othercomputers/My Mac/`.



**Already frozen locally — never rebuild here:** MB-10xx/11xx/20xx/30xx/61xx/62xx/63xx/70xx/8xxx/9xxx.



Recipe: Zhavoronkov et al., *Cell* 189:5980–5994 (2026).


## 0 · Mount + auto-discover (Othercomputers/My Mac)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

# Drive for Desktop computers-sync mounts at /content/drive/Othercomputers/My Mac/
# (read-only from Colab); mirrored copies may also exist under MyDrive.
SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']

def find_dir(target_name, search_roots=None, depth=6):
    if search_roots is None:
        search_roots = SEARCH_ROOTS
    matches = []
    for root in search_roots:
        direct = os.path.join(root, target_name)
        if os.path.isdir(direct):
            matches.append(direct)
    if matches:
        return matches
    def walk(path, d):
        if d > depth:
            return
        try:
            for entry in os.scandir(path):
                if entry.is_dir(follow_symlinks=False):
                    if entry.name == target_name:
                        matches.append(entry.path)
                    walk(entry.path, d + 1)
        except (PermissionError, OSError):
            pass
    for root in search_roots:
        if os.path.exists(root):
            walk(root, 0)
    return matches

cands = find_dir('masld-llm')
rev   = find_dir('masld-revision')
git_  = find_dir('masld-clinical-transcriptomics-github')
print('masld-llm candidates:');        [print('  ', p) for p in cands]
print('masld-revision candidates:');   [print('  ', p) for p in rev]
print('github-folder candidates:');    [print('  ', p) for p in git_]

MASLD_LLM = min(cands, key=len) if cands else None
MASLD_REV = min(rev, key=len) if rev else None
MASLD_GIT = min(git_, key=len) if git_ else None

def is_writable(path):
    try:
        probe = os.path.join(path, '.write_test')
        open(probe, 'w').close(); os.remove(probe)
        return True
    except Exception:
        return False

if MASLD_LLM and is_writable(MASLD_LLM):
    OUT = os.path.join(MASLD_LLM, 'prompts')
else:
    OUT = '/content/drive/MyDrive/masld-llm-outputs/prompts'
os.makedirs(OUT, exist_ok=True)
print()
print('MASLD_LLM =', MASLD_LLM)
print('MASLD_REV =', MASLD_REV)
print('MASLD_GIT =', MASLD_GIT)
print('OUT (writable) =', OUT)

## 1 · Tahoe-100M perturbation builder (MB-91xx/92xx)

43 liver-duo DE shards; split by cell line (HCC-2998 + HCT15 held out).


In [ ]:
import glob, random, json, gzip, time
import pyarrow.parquet as pq
import pandas as pd

T0 = time.time()
rng = random.Random(20260930)
TEST_LINES = ['HCC-2998', 'HCT15']
TARGET_N = 60_000

shards = sorted(glob.glob(f'{MASLD_REV}/data/tahoe/pseudobulk_de/*.parquet'))
print('shards found:', len(shards))
assert shards, 'Tahoe shards not found - check MASLD_REV path'

rows_train, rows_test, scanned = [], [], 0
for shard in shards:
    t = pq.read_table(shard, columns=['gene_name','log2FoldChange','padj','drug','Cell_Name_Vevo','concentration'])
    df = t.to_pandas()
    del t
    scanned += len(df)
    df = df[df['padj'] < 0.05]
    is_test = df['Cell_Name_Vevo'].isin(TEST_LINES)
    take = min(len(df[~is_test]), TARGET_N // max(1, len(shards)))
    if take:
        rows_train.append(df[~is_test].sample(n=take, random_state=20260930))
    if is_test.any():
        rows_test.append(df[is_test].head(200))
    del df
    print(f'  {scanned:10d} rows scanned ({time.time()-T0:.0f}s', flush=True)

train = pd.concat(rows_train) if rows_train else pd.DataFrame()
test  = pd.concat(rows_test) if rows_test else pd.DataFrame()
print('train rows:', len(train), '| test rows:', len(test))

def write_jsonl(records, path, meta):
    meta = {'n_records': len(records), 'seed': 20260930, **meta}
    with gzip.open(path, 'wt') as f:
        f.write(json.dumps({'__meta__': meta}) + chr(10))
        for r in records:
            f.write(json.dumps(r) + chr(10))
    print('  wrote', path)

recs = [dict(task='MB-9101', split='train', cell_line=r.Cell_Name_Vevo, drug=r.drug,
             prompt=(f"In an in-vitro perturbation experiment, cell line {r.Cell_Name_Vevo} was treated with "
                     f"{r.drug} at {r.concentration}. Does expression of {r.gene_name} increase or decrease "
                     f"relative to control? Answer with increase or decrease."),
             answer='increase' if r.log2FoldChange > 0 else 'decrease')
        for r in train.itertuples()]
write_jsonl(recs, f'{OUT}/MB-91xx_tahoe_train.jsonl.gz',
            meta=dict(builder='Tahoe-100M liver shards', test_lines=TEST_LINES, n_scanned=scanned))
recs = [dict(task='MB-9201', split='test', cell_line=r.Cell_Name_Vevo, drug=r.drug,
             prompt=(f"In an in-vitro perturbation experiment, cell line {r.Cell_Name_Vevo} was treated with "
                     f"{r.drug} at {r.concentration}. Does expression of {r.gene_name} increase or decrease "
                     f"relative to control? Answer with increase or decrease."),
             answer='increase' if r.log2FoldChange > 0 else 'decrease')
        for r in test.itertuples()]
write_jsonl(recs, f'{OUT}/MB-92xx_tahoe_test.jsonl.gz',
            meta=dict(builder='Tahoe held-out cell lines', test_lines=TEST_LINES))
print(f'done in {time.time()-T0:.0f}s')

## 2 · Sync-back note

If OUT is the read-only fallback (MyDrive/masld-llm-outputs), copy the new

MB-91xx/92xx files into `~/masld-llm/prompts/` on your laptop once synced,

then update MANIFEST locally.
